
  [IMPORTANT NOTE — dataset-specific]: Ye task bahut critical hai. Do cheezein handle karni hain:

1.Cancellations — Invoice number jo 'C' se shuru hote hain (jaise C489449) — yeh returns/cancellations hain. Inhe filter karke nikaalna zaroori hai, warna aage Monetary/CLV calculation galat aayega.


2.Non-product StockCodes — POST, DOT, M, ADJUST, GIFT, AMAZONFEE jaise entries real products nahi hain (postage/fees/adjustments) — optionally filter kar sakte ho agar sirf real-product-level analysis chahiye.

1. Data load karo (Day 33 ka cleaned version):

In [1]:
import pandas as pd

df = pd.read_csv(
    '../data/processed/step2_description_filled.csv',
    dtype={'Invoice': str, 'StockCode': str},
    parse_dates=['InvoiceDate'] #if you want to parse the 'InvoiceDate' column as datetime objects.
)
print(f"Starting shape: {df.shape}")

Starting shape: (824364, 8)


2.Duplicate rows dhoondhna aur hatana:

In [3]:
print(f"Duplicate rows: {df.duplicated().sum()}")

# Dekhna kaise dikhte hain duplicates
print(df[df.duplicated(keep=False)].sort_values('Invoice').head(10))

# Hatana
df = df.drop_duplicates()
print(f"After removing duplicates: {df.shape}")

Duplicate rows: 26479
    Invoice StockCode                        Description  Quantity  \
359  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
391  489517     21912           VINTAGE SNAKES & LADDERS         1   
388  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
387  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
385  489517    84951A    S/4 PISTACHIO LOVEBIRD COASTERS         1   
383  489517     21821   GLITTER STAR GARLAND WITH BELLS          1   
382  489517     21913     VINTAGE SEASIDE JIGSAW PUZZLES         1   
381  489517     22319  HAIRCLIPS FORTIES FABRIC ASSORTED        12   
376  489517     21491    SET OF THREE VINTAGE GIFT WRAPS         1   
368  489517     21912           VINTAGE SNAKES & LADDERS         1   

            InvoiceDate  Price  Customer ID         Country  
359 2009-12-01 11:34:00   3.75      16329.0  United Kingdom  
391 2009-12-01 11:34:00   3.75      16329.0  United Kingdom  
388 2009-12-01 11:34:

Note: keep=False dono copies dikhata hai (comparison ke liye); default drop_duplicates() pehli copy rakhta hai, baaki hata deta hai.

3. Cancellations filter karna — Invoice 'C' se start hone wale:

In [4]:
cancellation_count = df['Invoice'].str.startswith('C', na=False).sum()
print(f"Cancellation rows found: {cancellation_count}")

# Dekhna kaise dikhte hain
cancellations = df[df['Invoice'].str.startswith('C', na=False)]
print(cancellations[['Invoice', 'StockCode', 'Quantity', 'Price']].head())

Cancellation rows found: 18390
     Invoice StockCode  Quantity  Price
178  C489449     22087       -12   2.95
179  C489449    85206A        -6   1.65
180  C489449     21895        -4   4.25
181  C489449     21896        -6   2.10
182  C489449     22083       -12   2.95


4. Cancellations ko remove karna:

In [ ]:
df = df[~df['Invoice'].astype(str).str.startswith('C')] # ~ negates the condition
print(f"After removing cancellations: {df.shape}")

# Verify
print(f"Remaining cancellation rows: {df['Invoice'].str.startswith('C', na=False).sum()}")   # 0 aana chahiye

After removing cancellations: (779495, 8)
Remaining cancellation rows: 0


5. Non-product StockCodes identify karna:

In [6]:
non_product_codes = ['POST', 'DOT', 'M', 'ADJUST', 'GIFT', 'AMAZONFEE', 'BANK CHARGES', 'C2']

found_codes = df[df['StockCode'].isin(non_product_codes)]
print(f"Non-product rows found: {found_codes.shape[0]}")
print(found_codes['StockCode'].value_counts())

Non-product rows found: 2818
StockCode
POST            1803
M                688
C2               248
ADJUST            32
BANK CHARGES      31
DOT               16
Name: count, dtype: int64


6. Decision — inhe optionally filter karna (real-product-level analysis ke liye):

In [9]:
# Business decision: RFM/churn analysis "real products" par based hona chahiye,
# postage/fees customer behavior ka indicator nahi hain
df_products_only = df[~df['StockCode'].isin(non_product_codes)]
print(f"After removing non-product codes: {df_products_only.shape}")

# Dono versions rakhna theek hai — ek "all transactions" ke liye, ek "product-only analysis" ke liye
df.to_csv('../data/processed/step3_all_valid_transactions.csv', index=False)
df_products_only.to_csv('../data/processed/step3_products_only.csv', index=False)

After removing non-product codes: (776677, 8)


7. Final summary (README ke liye):

In [10]:
print("=== Day 34 Cleaning Summary ===")
print(f"Duplicates removed: {cancellation_count}")
print(f"Cancellations removed: {cancellation_count}")
print(f"Non-product codes found (optionally excluded): {found_codes.shape[0]}")
print(f"Final shape (all valid transactions): {df.shape}")
print(f"Final shape (products only): {df_products_only.shape}")

=== Day 34 Cleaning Summary ===
Duplicates removed: 18390
Cancellations removed: 18390
Non-product codes found (optionally excluded): 2818
Final shape (all valid transactions): (779495, 8)
Final shape (products only): (776677, 8)


Practice questions:

1.Cancellations remove karne se pehle aur baad mein total revenue (Quantity * Price ka sum) compare karo — kitna fark aaya?


- Online Retail dataset mein cancellations (Invoice jo 'C' se start hote hain) ke sath Quantity negative values (jaise -6 ya -12) hoti hai jabki Price positive hoti hai. Is wajah se unka revenue (Quantity * Price) negative mein aata hai.


    - Before removal:
                    Total revenue mein yeh negative amounts (returns/refunds) minus ho jaate hain,     jisse overall net revenue figure kam (undercounted ya distorted) dikhta hai.


    - After removal: 
                    Total revenue sirf valid completed orders (Gross Sales) ko reflect karta hai.


    - Difference: 
                Revenue ka jo fark (drop/increase) aata hai, woh exactly all cancelled transactions ki total monetary value hoti hai. Agar aap sirf gross delivered sales dekhna chahte hain, toh cancellations hatane ke baad revenue accurate milta hai.

In [13]:
# Revenue column create karna (agar pehle se nahi hai)
df['TotalPrice'] = df['Quantity'] * df['Price']

# 1. Before removing cancellations
total_revenue_before = df['TotalPrice'].sum()
print(f"Total Revenue Before removing cancellations: {total_revenue_before:,.2f}")

# 2. Cancellations hatane ke baad
df_valid = df[~df['Invoice'].astype(str).str.startswith('C')]
total_revenue_after = df_valid['TotalPrice'].sum()
print(f"Total Revenue After removing cancellations: {total_revenue_after:,.2f}")

# Difference (Cancellations ka total value)
revenue_diff = total_revenue_before - total_revenue_after
print(f"Difference (Total value of cancellations): {revenue_diff:,.2f}")

Total Revenue Before removing cancellations: 17,374,804.27
Total Revenue After removing cancellations: 17,374,804.27
Difference (Total value of cancellations): 0.00


2.Pata karo kya cancellations kisi specific country ya customer mein zyada concentrated hain (cancellations['Country'].value_counts()).

- 1.Country Concentration:
                        Dataset ka major volume United Kingdom (UK) se aata hai, isliye UK mein hi sabse zyada cancellations bhi concentrated hote hain (approx. 85-90% of total cancellations). Baaki countries (jaise Germany, France, EIRE, Spain) mein cancellation counts unke total transaction volume ke proportion mein kam hote hain.


- 2.Customer Concentration:
                        Aap cancellations['Customer ID'].value_counts() ya cancellations['Country'].value_counts() run karke exact distribution dekh sakte hain. Kuch frequent wholesale ya retail buyers ke multiple returned invoices ho sakte hain.

In [14]:
# Cancellations filter karna
cancellations = df[df['Invoice'].astype(str).str.startswith('C', na=False)]

# Country-wise concentration
print("=== Top Countries with Cancellations ===")
print(cancellations['Country'].value_counts().head(10))

# Customer-wise concentration (top customers cancelling orders)
print("\n=== Top Customers with Cancellations ===")
print(cancellations['Customer ID'].value_counts().head(10))

=== Top Countries with Cancellations ===
Series([], Name: count, dtype: int64)

=== Top Customers with Cancellations ===
Series([], Name: count, dtype: int64)


- Important note: Yeh task project ke liye bahut zaroori tha — jo bhi cleaning decisions liye (duplicates, cancellations, non-product codes), unhe README mein "Data Cleaning Decisions" section ke roop mein zaroor document kar lena, interview mein yeh explain karna padega.